# 04 · Visualização da rede de colaboração UNIFEI — geral e por instituto

**Entradas** (em `DATA_DIR`): `nodes.csv` (pesquisador + instituto) e `edges.csv` (coautorias).

**Saídas** (em `OUT_DIR`):
1. `00_rede_geral_por_instituto.png` — maior componente conexo, colorido por instituto, com layout **ForceAtlas2**;
2. `<SIGLA>_rede.png` — um gráfico por instituto, só com colaboração **intra-instituto**;
3. `figuras_redes.zip` — todas as figuras.

**Decisões de visualização**
- As figuras são geradas **no tamanho final em que aparecem no PDF** (`FIG_GERAL` e `FIG_INST`), para que fontes de 6–7 pt continuem legíveis no LaTeX.
- Rede geral: tamanho do nó proporcional ao grau; instituto não identificado desenhado **por baixo**, em cinza claro e menor; institutos identificados por cima; 8 maiores hubs rotulados com nome curto.
- O layout ForceAtlas2 usa sementes fixas (`random` e `numpy`), mas pode variar um pouco entre execuções e versões da biblioteca.
- Não há título dentro da imagem: as informações (nº de pesquisadores, tamanho do nó, rótulos) vão na legenda do LaTeX.
- Para pesquisadores com mais de um instituto, a cor na rede geral usa o primeiro da lista.

In [ ]:
!pip install adjustText fa2_modified -q

In [ ]:
import os
import random
import zipfile
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from adjustText import adjust_text

try:                                   # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = os.environ.get("DATA_DIR", "/content/drive/MyDrive")
    OUT_DIR = os.environ.get("OUT_DIR", "/content")
except ImportError:                    # execução local
    DATA_DIR = os.environ.get("DATA_DIR", "data")
    OUT_DIR = os.environ.get("OUT_DIR", "figs")
os.makedirs(OUT_DIR, exist_ok=True)

nodes = pd.read_csv(os.path.join(DATA_DIR, "nodes.csv"))
edges = pd.read_csv(os.path.join(DATA_DIR, "edges.csv"))

# ----------------------------------------------------------------------
# PARÂMETROS DE TAMANHO (em polegadas) = tamanho final no artigo
#   Uma coluna do template: ~3.4 de largura | Página inteira (figure*): ~7.0
# ----------------------------------------------------------------------
FIG_GERAL = (7.0, 5.2)    # Figura 1 (se ficar em uma coluna: (3.4, 3.1))
FIG_INST  = (3.4, 2.8)    # Figuras por instituto (se for figure*: (7.0, 5.0))
DPI = 300
N_ROTULOS_GERAL = 8       # hubs rotulados na rede geral
N_ROTULOS_INST  = 8       # hubs rotulados em cada instituto
SEED = 42

INSTITUTOS = ["IEM", "IEPG", "IESTI", "IFQ", "IMC", "IRN", "ISEE", "IEI"]

CORES = {
    "IEM":  "#e6194B",
    "IEPG": "#3cb44b",
    "IESTI":"#4363d8",
    "IFQ":  "#f58231",
    "IMC":  "#911eb4",
    "IRN":  "#42d4f4",
    "ISEE": "#f032e6",
    "IEI":  "#9acd32",
    "Não identificado": "#d9d9d9",
}


def instituto_principal(valor):
    """Se o pesquisador tem mais de um instituto (separado por ';'),
    usa o primeiro apenas para a cor no grafo geral."""
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não identificado"
    return str(valor).split(";")[0].strip()


def curto(nome):
    """Nome curto para rótulo. Ex.: 'Geraldo Lúcio Tiago Filho' -> 'G. Tiago Filho'."""
    p = str(nome).replace(".", ". ").split()
    if len(p) < 2:
        return str(nome)
    sufixos = {"filho", "junior", "júnior", "neto", "sobrinho"}
    ultimo = " ".join(p[-2:]) if len(p) > 2 and p[-1].lower() in sufixos else p[-1]
    return f"{p[0][0]}. {ultimo}".title()


nodes["instituto_cor"] = nodes["instituto"].apply(instituto_principal)
rotulo_de = dict(zip(nodes["id"], nodes["label"]))

print("Nós:", len(nodes), "| Arestas:", len(edges))

## 1. Gráfico geral (maior componente, layout ForceAtlas2)

In [ ]:
G = nx.Graph()
for _id, lab, inst in zip(nodes["id"], nodes["label"], nodes["instituto_cor"]):
    G.add_node(_id, label=lab, instituto=inst)
for s, t, w in zip(edges["source"], edges["target"], edges["weight"]):
    G.add_edge(s, t, weight=w)

# Só o maior componente: deixa o layout muito mais legível
componentes = sorted(nx.connected_components(G), key=len, reverse=True)
print("Maior componente:", len(componentes[0]), "de", G.number_of_nodes(), "nós")
print("Número de componentes:", len(componentes))
print("Nós isolados:", sum(1 for c in componentes if len(c) == 1))

Gc = G.subgraph(componentes[0]).copy()
graus = dict(Gc.degree())

# --- Layout ForceAtlas2 (500 iterações, ~30 s) ------------------------
random.seed(SEED); np.random.seed(SEED)
try:
    from fa2_modified import ForceAtlas2
    fa = ForceAtlas2(outboundAttractionDistribution=True, linLogMode=False,
                     adjustSizes=False, edgeWeightInfluence=1.0, jitterTolerance=1.0,
                     barnesHutOptimize=True, barnesHutTheta=1.2, scalingRatio=2.0,
                     strongGravityMode=False, gravity=1.0, verbose=False)
    pos = fa.forceatlas2_networkx_layout(Gc, pos=None, iterations=500)
except ImportError:
    print("fa2_modified indisponível: usando spring_layout como alternativa.")
    pos = nx.spring_layout(Gc, k=1.4 / np.sqrt(Gc.number_of_nodes()), iterations=100, seed=SEED)

nao_id = [n for n in Gc.nodes() if Gc.nodes[n]["instituto"] == "Não identificado"]
ident  = [n for n in Gc.nodes() if Gc.nodes[n]["instituto"] != "Não identificado"]

fig, ax = plt.subplots(figsize=FIG_GERAL, dpi=DPI)

nx.draw_networkx_edges(Gc, pos, ax=ax, alpha=0.05, width=0.2, edge_color="gray")

# Não identificados: por baixo, pequenos e claros
nx.draw_networkx_nodes(Gc, pos, nodelist=nao_id, ax=ax, node_color=CORES["Não identificado"],
                       node_size=[2 + 0.25 * graus[n] for n in nao_id],
                       linewidths=0, alpha=0.7)

# Identificados: por cima, maiores e coloridos
nx.draw_networkx_nodes(Gc, pos, nodelist=ident, ax=ax,
                       node_color=[CORES[Gc.nodes[n]["instituto"]] for n in ident],
                       node_size=[5 + 0.6 * graus[n] for n in ident],
                       linewidths=0.2, edgecolors="black", alpha=0.95)

top_hubs = sorted(graus, key=graus.get, reverse=True)[:N_ROTULOS_GERAL]
texts = []
for n in top_hubs:
    x, y = pos[n]
    texts.append(ax.text(x, y, curto(Gc.nodes[n]["label"]), fontsize=6, fontweight="bold",
                         bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=0.3)))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="black", lw=0.4))

handles = [mpatches.Patch(color=CORES[i], label=i) for i in INSTITUTOS + ["Não identificado"]]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=5,
          fontsize=6.5, frameon=False, handlelength=1, columnspacing=0.8)
ax.axis("off")
plt.savefig(os.path.join(OUT_DIR, "00_rede_geral_por_instituto.png"), dpi=DPI, bbox_inches="tight")
plt.show()

## 2. Um gráfico por instituto (só colaboração intra-instituto)

Usa **apenas as arestas entre pesquisadores do mesmo instituto**. Pesquisadores sem coautoria dentro do próprio instituto ficam fora do desenho (e são contados em `isolados`).

In [ ]:
for sigla in INSTITUTOS:

    ids_instituto = set(nodes.loc[nodes["instituto"].str.contains(sigla, na=False), "id"])
    sub_edges = edges[edges["source"].isin(ids_instituto) & edges["target"].isin(ids_instituto)]

    Gi = nx.Graph()
    for _id in ids_instituto:
        Gi.add_node(_id, label=rotulo_de[_id])
    for s, t, w in zip(sub_edges["source"], sub_edges["target"], sub_edges["weight"]):
        Gi.add_edge(s, t, weight=w)

    isolados = list(nx.isolates(Gi))
    Gi.remove_nodes_from(isolados)

    if Gi.number_of_nodes() == 0:
        print(f"{sigla}: sem colaborações intra-instituto, pulando.")
        continue

    # kamada_kawai separa melhor clusters em grafos menores;
    # spring_layout é mais rápido/estável para os maiores
    if Gi.number_of_nodes() <= 220:
        try:
            pos_i = nx.kamada_kawai_layout(Gi)
        except Exception:
            pos_i = nx.spring_layout(Gi, k=2.2 / np.sqrt(Gi.number_of_nodes()), iterations=150, seed=7)
    else:
        pos_i = nx.spring_layout(Gi, k=2.2 / np.sqrt(Gi.number_of_nodes()), iterations=150, seed=7)

    graus_i = dict(Gi.degree())

    fig, ax = plt.subplots(figsize=FIG_INST, dpi=DPI)

    node_sizes_i = [8 + 3 * graus_i[n] for n in Gi.nodes()]
    edge_weights = [Gi[u][v]["weight"] for u, v in Gi.edges()]
    max_w = max(edge_weights) if edge_weights else 1
    edge_widths = [0.2 + 1.0 * (w / max_w) for w in edge_weights]

    nx.draw_networkx_edges(Gi, pos_i, ax=ax, alpha=0.3, width=edge_widths, edge_color="#888888")
    nx.draw_networkx_nodes(Gi, pos_i, ax=ax, node_color=CORES[sigla], node_size=node_sizes_i,
                           linewidths=0.3, edgecolors="black", alpha=0.95)

    n_rot = min(N_ROTULOS_INST, Gi.number_of_nodes())
    top_n = sorted(graus_i, key=graus_i.get, reverse=True)[:n_rot]

    texts_i = []
    for n in top_n:
        x, y = pos_i[n]
        texts_i.append(ax.text(x, y, curto(Gi.nodes[n]["label"]), fontsize=5.5,
                               bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=0.2)))
    adjust_text(texts_i, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))

    # Sem título na imagem: as informações vão na legenda do LaTeX
    ax.axis("off")
    plt.savefig(os.path.join(OUT_DIR, f"{sigla}_rede.png"), dpi=DPI, bbox_inches="tight")
    plt.show()

    print(f"{sigla}: {Gi.number_of_nodes()} pesquisadores conectados, "
          f"{Gi.number_of_edges()} colaborações, {len(isolados)} sem coautoria interna (ocultos)\n")

## 3. Reunir todas as imagens em um .zip

In [ ]:
zip_path = os.path.join(OUT_DIR, "figuras_redes.zip")
arquivos = ["00_rede_geral_por_instituto.png"] + [f"{s}_rede.png" for s in INSTITUTOS]

with zipfile.ZipFile(zip_path, "w") as z:
    for nome in arquivos:
        caminho = os.path.join(OUT_DIR, nome)
        if os.path.exists(caminho):
            z.write(caminho, nome)
        else:
            print(f"Aviso: {nome} não encontrado, ignorando.")

try:                                   # no Colab, baixa o .zip
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Figuras salvas em:", os.path.abspath(OUT_DIR))